# Part 1: Variational Autoencoder (VAE)
## Learning to Compress and Generate Images

In our captioning notebook, we built a model that:
- **Encodes** an image into a sequence of feature vectors (ResNet)
- **Decodes** those features into text (Transformer decoder)

A VAE is structurally similar but the task is different:
- **Encodes** an image into a compact **latent vector** z
- **Decodes** z back into a reconstructed image

The key insight that makes a VAE *generative*: instead of encoding to a single
fixed vector, we encode to a **probability distribution** — a mean μ and variance σ².
We then *sample* z from that distribution. At generation time, we skip the encoder
entirely and sample z directly from a standard normal distribution N(0,1),
then decode it into a new image.

### Connections to what we've built

| Captioning Model | VAE |
|---|---|
| ResNet encoder → (batch, 49, d_model) | CNN encoder → (batch, latent_dim) |
| Transformer decoder → text tokens | CNN decoder → pixel values |
| Cross-attention bridges encoder→decoder | z vector bridges encoder→decoder |
| Trained with cross-entropy loss | Trained with reconstruction + KL loss |
| Can't generate without an input image | Can generate by sampling z ~ N(0,1) |

### The training objective

VAE loss = **Reconstruction loss** + **KL divergence**

- **Reconstruction loss:** how well does the decoder reproduce the original image?
  (pixel-level MSE or BCE — same idea as our caption cross-entropy loss)
- **KL divergence:** how close is the learned distribution q(z|x) to N(0,1)?
  This regularizes the latent space so it stays structured and sampleable.

The tension between these two terms is what makes a VAE interesting:
reconstruction wants to use the latent space freely; KL wants to compress
it toward a standard normal. The balance produces a smooth, continuous
latent space where nearby points decode to similar images.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import numpy as np
import matplotlib.pyplot as plt
import time
import os

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: cuda


---
## 1. Dataset: CelebA Faces (64×64)

We'll train on CelebA — 200,000 celebrity face images. Using 64×64 resolution
keeps things tractable. The VAE will learn to compress each face into a
latent vector and reconstruct it, and we can sample new faces at inference time.

Compare to our captioning notebook: there we had 40,000 (image, caption) pairs.
Here we just have 200,000 images — no labels, no captions. This is
**unsupervised learning**: the only supervision is the image itself.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
import gdown

# Define the data directory
DATA_DIR = '/content/drive/MyDrive/Colab/data'
CHECKPOINT_DIR = '/content/drive/MyDrive/Colab/checkpoints'
ZIP_FILE = 'celeba.zip'
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# Google Drive file URL
file_id = '1sOmeLrLNmqdRlM6imC6YeQ2SWFKHWww_'
output_path = os.path.join(DATA_DIR, ZIP_FILE)

# Download only if the file does not exist
if not os.path.exists(output_path):
    print(f'Downloading file to {output_path}...')
    gdown.download(id=file_id, output=output_path, quiet=False)
    print('Download complete!')
else:
    print(f'File already exists at {output_path}. Skipping download.')

File already exists at /content/drive/MyDrive/Colab/data/celeba.zip. Skipping download.


In [ ]:
import shutil
import zipfile
import os

# Define the data path
ZIPPED_DATA_PATH = '/content/celeba.zip'
UNZIPPED_DATA_PATH = '/content/celeba_unzipped'

# 1. Copy the heavy zip file from Drive to fast local storage
if not os.path.exists(ZIPPED_DATA_PATH):
  print("Copying zip file to local storage...")
  shutil.copy(f"{DATA_DIR}/{ZIP_FILE}", ZIPPED_DATA_PATH)
else:
  print(f'Zip file already exists at {ZIPPED_DATA_PATH}. Skipping copy.')

# 2. Extract it locally, only if the directory doesn't exist
if not os.path.exists(UNZIPPED_DATA_PATH):
    print("Extracting zip file to folder of images...")
    with zipfile.ZipFile(ZIPPED_DATA_PATH, 'r') as zip_ref:
        zip_ref.extractall(UNZIPPED_DATA_PATH)
    print("Done!")
else:
    print(f'Unzipped data already exists at {UNZIPPED_DATA_PATH}. Skipping extraction.')

Zip file already exists at /content/celeba.zip. Skipping copy.
Unzipped data already exists at /content/celeba_unzipped. Skipping extraction.


In [ ]:
import os
import glob
from PIL import Image
from torch.utils.data import Dataset, random_split
import sys

IMAGE_SIZE = 64

transform = transforms.Compose([
    # Ensure all images are 64x64
    transforms.Resize(IMAGE_SIZE), ## resize smaller dimension to 64px, maintain aspect ratio
    transforms.CenterCrop(IMAGE_SIZE),  ## crop out 64x64 center of image

    transforms.ToTensor(), # scale all RGBs to [0,1] and convert to pytorch tensor
    transforms.Normalize(mean=[0.5, 0.5, 0.5],
                         std=[0.5, 0.5, 0.5]),   # rescale to [-1, 1] by subtracting mean and dividing by std
])

# Custom Dataset for loading images, UN-labeled
class SimpleCelebA(Dataset):
    def __init__(self, root_dir, transform=None, scale=1.0):
        # Find all jpg files anywhere in the root directory
        self.image_paths = sorted(glob.glob(os.path.join(root_dir, '**', '*.jpg'), recursive=True))
        size = int(len(self.image_paths) * scale)
        self.image_paths = self.image_paths[:size]
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img = Image.open(self.image_paths[idx]).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, 0  # Return dummy label 0 because we're buildling an auto-en

In [ ]:
BATCH_SIZE = 256

# 3. Update the path and disable caching
full_dataset = SimpleCelebA(UNZIPPED_DATA_PATH, transform=transform, scale=1.0)

# Split into 90% train, 10% val
train_size = int(0.9 * len(full_dataset))
val_size = len(full_dataset) - train_size

train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

# Maximize data loading performance
import os
num_workers = os.cpu_count() or 2
print(f"Using {num_workers} workers")
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE,
                          shuffle=True, num_workers=num_workers, pin_memory=True, prefetch_factor=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE,
                        shuffle=False, num_workers=num_workers, pin_memory=True, prefetch_factor=2)

print(f'Train images: {len(train_dataset):,}')
print(f'Val images:   {len(val_dataset):,}')

# Visualize some examples
images, _ = next(iter(train_loader))
fig, axes = plt.subplots(2, 8, figsize=(16, 4))
for i, ax in enumerate(axes.flatten()):
    img = images[i].permute(1, 2, 0).numpy() * 0.5 + 0.5
    ax.imshow(img.clip(0, 1))
    ax.axis('off')
plt.suptitle('CelebA Training Images (64×64)', fontsize=12)
plt.tight_layout()
plt.show()

---
## 2. The Encoder

The encoder compresses a 64×64×3 image into two vectors:
- **μ (mu):** the mean of the latent distribution
- **log σ² (log_var):** the log-variance of the latent distribution

We use log-variance instead of σ² directly because it's unconstrained
(can be any real number), making optimization easier.

Compare to our ResNet encoder in the captioning notebook: that encoder
produced 49 feature vectors, one per spatial region. This encoder produces
just TWO vectors (μ and log σ²), each of dimension `latent_dim`.
It's a much more aggressive compression.

Architecture: Conv → Conv → Conv → Conv → Flatten → Linear to μ, Linear to log σ²

Each conv layer halves the spatial size (stride=2), so:
```
64×64 → 32×32 → 16×16 → 8×8 → 4×4 → flatten → latent_dim
```

In [ ]:
class Encoder(nn.Module):
    """
    CNN encoder: image → (mu, log_var)

    Produces the parameters of a Gaussian distribution q(z|x).
    The actual latent vector z is sampled from this distribution
    in the VAE's forward method (the reparameterization trick).
    """
    def __init__(self, latent_dim, channels=[32, 64, 128, 256]):
        super().__init__()
        # Convolutional feature extractor
        # Each conv: halves spatial size (stride=2), increases channels
        self.conv_layers = nn.Sequential(

            ## TODO: Create 4 Convolutional 2D layers, using the channel sizes passed in as parameters.
            ## The first layer goes from 3 channels to channels[0], the second from channels[0] to channels[1]
            ## And so on
            ## Kernel size = 4, stride = 2, padding = 1 in all cases
            ## each layer is followed by BatchNord2d(in_channels) and LeakyReLU(0.2)

        )
        # After 4 stride-2 convolutions: 64 → 4 spatial size
        # Flattened size: channels[-1] * 4 * 4
        self.flat_dim = channels[-1] * 4 * 4

        # Two separate linear heads: one for mu, one for log_var
        # Both map from flat_dim → latent_dim
        # "fc" stands for "fully connected"
        self.fc_mu = ## TODO: define a linear layer from size self.flat_dim to size latent_dim
        self.fc_log_var = ## TODO: make a layer like the one for fc_mu

    def forward(self, x):
        """
        x: (batch, 3, 64, 64)
        returns: mu (batch, latent_dim), log_var (batch, latent_dim)
        """

        ##TODO: fill in missing lines described below

        # let h = the result of passing 'x' to the convolution layers defined before
        # let h = h.view(h.size(0), -1) -- this reshapes the output 'g' to a batch of 1D vectors
        # let mu = the result of passing h to the fully connected 'mu' layer
        # let log_var = the result of passing h to the fully connected 'log_var' layer

        # Note the sizes of each layer output:
        # (batch, 256, 4, 4) where 256 = num of output channels and 4x4 is size of last layer
        # (batch, 256*4*4)
        # (batch, latent_dim)
        # (batch, latent_dim)

        return mu, log_var

---
## 3. The Reparameterization Trick

We need to **sample** z from the distribution N(μ, σ²). But sampling is not
differentiable — the gradient can't flow through a random sample.

The trick: instead of sampling z ~ N(μ, σ²) directly, we write:

**z = μ + σ · ε**,  where ε ~ N(0, 1)

Now the randomness (ε) is in a fixed distribution we don't differentiate
through. The parameters μ and σ are computed by the encoder and ARE
differentiable. Gradients can flow through μ and σ to the encoder.

This is a beautiful trick — moving the stochasticity out of the computational
graph so backprop still works.

Compare to our attention notebooks: we never needed this because attention
is deterministic. The VAE introduces randomness deliberately, and this
trick is how we handle it during training.

In [ ]:
def reparameterize(mu, log_var):
    """
    Sample z from N(mu, sigma^2) using the reparameterization trick.

    z = mu + sigma * epsilon,  where epsilon ~ N(0, 1)

    We compute sigma = exp(0.5 * log_var) since log_var = log(sigma^2),
    so exp(0.5 * log_var) = exp(log(sigma)) = sigma.

    mu:      (batch, latent_dim)
    log_var: (batch, latent_dim)
    returns: z (batch, latent_dim)
    """
    if not torch.is_grad_enabled():
        # During inference, just use the mean (deterministic)
        return mu

    std = torch.exp(0.5 * log_var)          # sigma = exp(0.5 * log_var)
    epsilon = torch.randn_like(std)          # epsilon ~ N(0,1), same shape as std
    # TODO: let result = mu + sigma * epsilon and return it

---
## 4. The Decoder

The decoder is the mirror image of the encoder:
- Input: z vector of shape (batch, latent_dim)
- Output: reconstructed image (batch, 3, 64, 64)

We use **transposed convolutions** (sometimes called deconvolutions) to
upsample from 4×4 back to 64×64. Each transposed conv doubles the spatial size:

```
latent_dim → Linear → 4×4 → 8×8 → 16×16 → 32×32 → 64×64
```

Compare to the captioning notebook's decoder: that used self-attention +
cross-attention to generate text tokens sequentially. This decoder uses
transposed convolutions to upsample spatially in parallel — image generation
doesn't have a natural sequential ordering, so we generate all pixels at once.

The final activation is **Tanh**, outputting values in [-1, 1] to match
the [-1, 1] normalization of our input images.

In [ ]:
class Decoder(nn.Module):
    """
    CNN decoder: z → reconstructed image

    Mirror of the encoder:
    Encoder: 64×64 image → ... convolutions ... → latent vector
    Decoder: latent vector → ... transposed convolutions ... → 64×64 image
    """
    def __init__(self, latent_dim, channels=[256, 128, 64, 32]):
        super().__init__()
        self.flat_dim = channels[0] * 4 * 4

        # Project from latent_dim to the spatial feature volume
        self.fc = nn.Linear(latent_dim, self.flat_dim)

        # Transposed convolutions to upsample back to 64×64
        # Each ConvTranspose2d doubles the spatial size (stride=2)
        self.deconv_layers = nn.Sequential(
            nn.ConvTranspose2d(channels[0], channels[1], kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(channels[1]),
            nn.ReLU(),

            # TODO: Add 2 more upscaling ConvTranspose2d layers just like the previous
            # Watch out for input/output sizes

            nn.ConvTranspose2d(channels[3], 3, kernel_size=4, stride=2, padding=1),
            nn.Tanh(),   # output in [-1, 1] to match normalized input
        )

    def forward(self, z):
        """
        z: (batch, latent_dim)
        returns: (batch, 3, 64, 64)
        """
        h = self.fc(z)                           # (batch, 256*4*4)
        h = h.view(h.size(0), -1, 4, 4)          # (batch, 256, 4, 4)
        return self.deconv_layers(h)              # (batch, 3, 64, 64)

---
## 5. The VAE

Assemble encoder + reparameterization + decoder.

Compare to our `ImageCaptioner` model which called `encode_image()` then `decode()`.
Same pattern here: `encode()` then `decode()`, with reparameterization in between.

In [ ]:
class VAE(nn.Module):
    """
    Variational Autoencoder.

    forward() returns the reconstructed image AND the distribution parameters
    (mu, log_var) needed for the KL divergence loss term.

    Compare to our ImageCaptioner.forward() which returned logits.
    We return both x_recon (for reconstruction loss) and mu/log_var (for KL loss).
    """
    def __init__(self, latent_dim=128):
        super().__init__()
        self.latent_dim = latent_dim
        self.encoder = Encoder(latent_dim)
        self.decoder = Decoder(latent_dim)

    def forward(self, x):
        """
        x:       (batch, 3, 64, 64)
        returns: x_recon (batch, 3, 64, 64), mu (batch, latent_dim), log_var (batch, latent_dim)
        """
        mu, log_var = self.encoder(x)
        z = reparameterize(mu, log_var)
        x_recon = self.decoder(z)
        return x_recon, mu, log_var

    @torch.no_grad()
    def sample(self, n_samples):
        """
        Generate new images by sampling z ~ N(0, 1) and decoding.
        No encoder needed — this is pure generation.

        Compare to our NMT model's greedy_translate() and CharGPT's generate():
        those also bypassed the encoder at inference time and generated autoregressively.
        Here we generate all pixels at once (non-autoregressive).
        """
        self.eval() ## this line just sets pytorch to EVAL mode instead of TRAIN mode -- it doesn't really do anything else
        z = torch.randn(n_samples, self.latent_dim, device=next(self.parameters()).device)
        return self.decoder(z)              # (n_samples, 3, 64, 64)

    @torch.no_grad()
    def reconstruct(self, x):
        """Encode then decode — used for visualization."""
        self.eval() ## this line just sets pytorch to EVAL mode instead of TRAIN mode -- it doesn't really do anything else
        mu, log_var = self.encoder(x)
        z = reparameterize(mu, log_var)
        return self.decoder(z)


# Instantiate and verify shapes
LATENT_DIM = 128
model = VAE(latent_dim=LATENT_DIM).to(device)

total = sum(p.numel() for p in model.parameters())
print(f'Total parameters: {total:,}')

# Shape check
_x = torch.randn(4, 3, 64, 64).to(device)
_recon, _mu, _log_var = model(_x)
print(f'Input:     {_x.shape}')
print(f'Recon:     {_recon.shape}')       # (4, 3, 64, 64)
print(f'Mu:        {_mu.shape}')           # (4, 128)
print(f'Log var:   {_log_var.shape}')      # (4, 128)
del _x, _recon, _mu, _log_var

---
## 6. The VAE Loss

The loss has two terms:

**Reconstruction loss:** How accurately does the decoder reproduce the input?
We use MSE (mean squared error) over pixels. This is analogous to the
cross-entropy loss in our captioning model — there we measured how well the
model predicted the correct words; here we measure how well it predicts pixels.

**KL divergence:** How much does the learned distribution q(z|x) = N(μ, σ²)
differ from the prior p(z) = N(0, 1)? The KL divergence has a closed form
for two Gaussians:

**KL = -0.5 · Σ(1 + log σ² - μ² - σ²)**

This is negative — minimizing -KL means minimizing the distance to N(0,1).

The **β** hyperparameter (beta-VAE) scales the KL term:
- β=1: standard VAE
- β>1: forces a more compressed, regular latent space
  (better for interpolation, worse reconstruction)
- β<1: allows more freedom (better reconstruction, messier latent space)

In [ ]:
def vae_loss(x, x_recon, mu, log_var, beta=1.0):
    """
    VAE loss = reconstruction loss + beta * KL divergence

    x:       (batch, 3, 64, 64) — original images
    x_recon: (batch, 3, 64, 64) — reconstructed images
    mu:      (batch, latent_dim)
    log_var: (batch, latent_dim)
    beta:    weight on KL term

    Compare to our captioning training:
        loss = F.cross_entropy(logits, targets, ignore_index=PAD)
    Here we have two terms — reconstruction (like cross_entropy)
    and KL (a regularizer specific to VAEs).
    """
    # Reconstruction loss: MSE between original and reconstructed pixels
    # Sum over pixels, mean over batch
    recon_loss = F.mse_loss(x_recon, x, reduction='sum') / x.size(0)

    # KL divergence: closed form for q(z|x) = N(mu, sigma^2) vs p(z) = N(0,1)
    # KL = -0.5 * sum(1 + log_var - mu^2 - exp(log_var))
    # Mean over batch
    kl_loss = -0.5 * torch.sum(1 + log_var - mu.pow(2) - log_var.exp()) / x.size(0)

    total_loss = recon_loss + beta * kl_loss
    return total_loss, recon_loss.item(), kl_loss.item()

---
## 7. Training

Same training loop structure as our captioning notebook:
- `train_epoch()` iterates over batches, computes loss, backprops
- `evaluate()` computes validation loss
- Main loop tracks losses and saves checkpoints

One difference: we use **KL annealing** — multiply the KL divergence term by $\beta$. Start with $\beta$=0 (reconstruction only)
and gradually increase $\beta$ to 1. This helps the model learn to reconstruct well
first before the KL term compresses the latent space. Without annealing, the
KL term can dominate early and collapse the latent space before the decoder
learns anything useful.

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

def get_beta(epoch, warmup_epochs=10, max_beta=1.0):
    """Linear KL annealing: beta goes from 0 to max_beta over warmup_epochs."""
    return min(max_beta, max_beta * epoch / warmup_epochs)

from tqdm.auto import tqdm

def train_epoch(model, loader, optimizer, beta):
    model.train()
    total_loss = recon_total = kl_total = 0

    # Wrap the loader with tqdm for a progress bar
    for images, _ in tqdm(loader, desc=f"Epoch {epoch}", leave=False):
        # Use non_blocking=True to transfer data asynchronously since pin_memory=True
        images = images.to(device, non_blocking=True)

        x_recon, mu, log_var = model(images)
        loss, recon, kl = vae_loss(images, x_recon, mu, log_var, beta=beta)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item()
        recon_total += recon
        kl_total += kl

    n = len(loader)
    return total_loss / n, recon_total / n, kl_total / n


@torch.no_grad()
def evaluate(model, loader, beta):
    model.eval()
    total_loss = recon_total = kl_total = 0

    for images, _ in loader:
        images = images.to(device, non_blocking=True)
        x_recon, mu, log_var = model(images)
        loss, recon, kl = vae_loss(images, x_recon, mu, log_var, beta=beta)
        total_loss += loss.item()
        recon_total += recon
        kl_total += kl

    n = len(loader)
    return total_loss / n, recon_total / n, kl_total / n

In [ ]:
def show_samples(model, n=8, title=''):
    """Generate and display random samples from the VAE."""
    samples = model.sample(n).cpu()
    fig, axes = plt.subplots(1, n, figsize=(2 * n, 2))
    for i, ax in enumerate(axes):
        img = samples[i].permute(1, 2, 0).numpy() * 0.5 + 0.5
        ax.imshow(img.clip(0, 1))
        ax.axis('off')
    if title:
        fig.suptitle(title, fontsize=11)
    plt.tight_layout()
    plt.show()


def show_reconstructions(model, loader, n=8, title=''):
    """Show original images next to their reconstructions."""
    images, _ = next(iter(loader))
    images = images[:n].to(device)
    recons = model.reconstruct(images).cpu()
    images = images.cpu()

    fig, axes = plt.subplots(2, n, figsize=(2 * n, 4))
    for i in range(n):
        for row, img_batch in enumerate([images, recons]):
            img = img_batch[i].permute(1, 2, 0).numpy() * 0.5 + 0.5
            axes[row, i].imshow(img.clip(0, 1))
            axes[row, i].axis('off')
    axes[0, 0].set_title('Original', fontsize=9)
    axes[1, 0].set_title('Reconstructed', fontsize=9)
    if title:
        fig.suptitle(title, fontsize=11)
    plt.tight_layout()
    plt.show()

## 7B. The Training Loop

In [ ]:
import sys
import os
import time
import torch

# Suppress the harmless multiprocessing AssertionError during DataLoader teardown
original_unraisablehook = sys.unraisablehook
def custom_unraisablehook(unraisable):
    if issubclass(unraisable.exc_type, AssertionError) and "can only test a child process" in str(unraisable.exc_value):
        return # Ignore this specific error
    original_unraisablehook(unraisable)
sys.unraisablehook = custom_unraisablehook

NUM_EPOCHS = 20
WARMUP_EPOCHS = 5
CHECKPOINT_PATH = f"{CHECKPOINT_DIR}/vae_celeba_best_00.pt"

counter = 0
while (os.path.exists(CHECKPOINT_PATH) and counter < 99):
    counter += 1
    CHECKPOINT_PATH = f"{CHECKPOINT_DIR}/vae_celeba_best_{counter:02}.pt"
if (counter == 99):
  print("Overwriting checkpoint #99. You should clean your checkpoint directory")

train_losses, val_losses = [], []
best_val_loss = float('inf')

print(f'Training VAE for {NUM_EPOCHS} epochs...')
print(f'KL annealing: beta goes from 0 → 1 over first {WARMUP_EPOCHS} epochs')
print(f"Writing Checkpoints to {CHECKPOINT_PATH}")

print()

start = time.time()

for epoch in range(1, NUM_EPOCHS + 1):
    beta = get_beta(epoch, WARMUP_EPOCHS)

    train_loss, train_recon, train_kl = train_epoch(model, train_loader, optimizer, beta)
    val_loss, val_recon, val_kl = evaluate(model, val_loader, beta)

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    elapsed = time.time() - start
    print(f'Epoch {epoch:2d}/{NUM_EPOCHS} | beta={beta:.2f} | '
          f'Train: {train_loss:.1f} (recon={train_recon:.1f} kl={train_kl:.1f}) | '
          f'Val: {val_loss:.1f} | {elapsed:.0f}s')

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), CHECKPOINT_PATH)
        print(f'  → New best saved')

    # Show progress every 5 epochs
    if epoch % 5 == 0:
        show_reconstructions(model, val_loader, title=f'Reconstructions at epoch {epoch}')
        show_samples(model, title=f'Generated samples at epoch {epoch}')

print(f'\nDone in {(time.time()-start)/60:.1f} minutes')

In [ ]:
# Optional: Continue training for more epochs
ADDITIONAL_EPOCHS = 0

print(f'Continuing training for {ADDITIONAL_EPOCHS} more epochs...')
print(f'Current epoch: {NUM_EPOCHS}, Warmup epochs: {WARMUP_EPOCHS} (Warmup is {"over" if NUM_EPOCHS >= WARMUP_EPOCHS else "not over"})')

start = time.time()

for epoch in range(NUM_EPOCHS + 1, NUM_EPOCHS + ADDITIONAL_EPOCHS + 1):
    beta = get_beta(epoch, WARMUP_EPOCHS)

    train_loss, train_recon, train_kl = train_epoch(model, train_loader, optimizer, beta)
    val_loss, val_recon, val_kl = evaluate(model, val_loader, beta)

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    elapsed = time.time() - start
    print(f'Epoch {epoch:2d}/{NUM_EPOCHS + ADDITIONAL_EPOCHS} | beta={beta:.2f} | '
          f'Train: {train_loss:.1f} (recon={train_recon:.1f} kl={train_kl:.1f}) | '
          f'Val: {val_loss:.1f} | {elapsed:.0f}s')

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), CHECKPOINT_PATH)
        print(f'  → New best saved')

    # Show progress every 5 epochs
    if epoch % 5 == 0:
        show_reconstructions(model, val_loader, title=f'Reconstructions at epoch {epoch}')
        show_samples(model, title=f'Generated samples at epoch {epoch}')

# Update total epochs run
NUM_EPOCHS += ADDITIONAL_EPOCHS

print(f'\nDone in {(time.time()-start)/60:.1f} minutes')

---
## 8. Training Curves

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(train_losses, label='Train', marker='o', markersize=4)
plt.plot(val_losses, label='Val', marker='s', markersize=4)
plt.axvline(x=WARMUP_EPOCHS - 1, color='gray', linestyle='--',
            alpha=0.5, label='KL warmup complete')
plt.xlabel('Epoch')
plt.ylabel('Total Loss')
plt.title('VAE Training Curves')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print('Note: loss changes trajectory when KL warmup kicks in — normal behavior.')
print('The model is learning to balance reconstruction quality and latent regularity.')

---
## 9. Load Best Model and Explore

In [ ]:
print("Loading saved checkpoint from " + CHECKPOINT_PATH)
model.load_state_dict(torch.load(CHECKPOINT_PATH, map_location=device))
model.eval()

# Reconstructions
show_reconstructions(model, val_loader, n=10, title='VAE Reconstructions (Best Model)')

# Random samples
show_samples(model, n=10, title='VAE Generated Samples (z ~ N(0,1))')

---
## 10. Latent Space Exploration

A cool demonstration of a VAE: the latent space is smooth and
continuous. We can interpolate between two faces and the decoder produces
plausible intermediate faces the whole way.

This is impossible with a standard autoencoder — without the KL term,
the latent space is full of holes and interpolations decode to garbage.
The KL term is what forces the space to be smooth.

In [ ]:
@torch.no_grad()
def interpolate(model, img1, img2, steps=5):
    """
    Interpolate between two images in latent space.
    Shows why the latent space is smooth: every point between z1 and z2
    decodes to a plausible image.
    """
    model.eval()
    img1_t = img1.unsqueeze(0).to(device)
    img2_t = img2.unsqueeze(0).to(device)

    # Encode both images to get their latent means
    mu1, _ = model.encoder(img1_t)
    mu2, _ = model.encoder(img2_t)

    # Linear interpolation in latent space
    alphas = torch.linspace(0, 1, steps, device=device)
    interpolated = []
    for alpha in alphas:
        z = (1 - alpha) * mu1 + alpha * mu2
        decoded = model.decoder(z).squeeze(0).cpu()
        interpolated.append(decoded)

    # Include originals at the beginning and end
    all_images = [img1.cpu()] + interpolated + [img2.cpu()]
    total_steps = len(all_images)

    # Display
    fig, axes = plt.subplots(1, total_steps, figsize=(2 * total_steps, 2.5))
    for i, (ax, img) in enumerate(zip(axes, all_images)):
        img_np = img.permute(1, 2, 0).numpy() * 0.5 + 0.5
        ax.imshow(img_np.clip(0, 1))
        ax.axis('off')
        if i == 0: ax.set_title('Orig 1', fontsize=8)
        elif i == 1: ax.set_title('z₁', fontsize=8)
        elif i == total_steps - 2: ax.set_title('z₂', fontsize=8)
        elif i == total_steps - 1: ax.set_title('Orig 2', fontsize=8)
    plt.suptitle('Latent Space Interpolation', fontsize=12)
    plt.tight_layout()
    plt.show()


# Grab two val images from the dataloader and interpolate
val_images, _ = next(iter(val_loader))
interpolate(model, val_images[100], val_images[101])
interpolate(model, val_images[100], val_images[5])
interpolate(model, val_images[100], val_images[20])

In [ ]:
@torch.no_grad()
def explore_latent_dimension(model, base_image, dim, n_steps=10, scale=3.0):
    """
    Vary a single latent dimension while holding others fixed.
    This shows what each dimension has learned to control.
    Some dimensions control pose, some lighting, some hair color, etc.
    """
    model.eval()
    base = base_image.unsqueeze(0).to(device)
    mu, _ = model.encoder(base)

    values = torch.linspace(-scale, scale, n_steps, device=device)
    images = []
    for v in values:
        z = mu.clone()
        z[0, dim] = v
        decoded = model.decoder(z).squeeze(0).cpu()
        images.append(decoded)

    fig, axes = plt.subplots(1, n_steps, figsize=(2 * n_steps, 2.5))
    for i, (ax, img) in enumerate(zip(axes, images)):
        img_np = img.permute(1, 2, 0).numpy() * 0.5 + 0.5
        ax.imshow(img_np.clip(0, 1))
        ax.axis('off')
    plt.suptitle(f'Varying latent dimension {dim}', fontsize=12)
    plt.tight_layout()
    plt.show()


# Explore the first several latent dimensions
# Some will be interpretable (hair, pose, gender, lighting)
# others will be entangled or less obvious
base_img = val_images[100]
for dim in range(80):
    explore_latent_dimension(model, base_img, dim=dim, scale = 10, n_steps = 7)

## Experiments

There are a number of critical parameters here in this notebook. Explore what happens when you make the following changes and train the model (you probably won't need all 20 epochs to see a result)

1. Set $\beta = 0$
2. Set $\beta = 1$
3. Set $\epsilon = 0$ in the reparamatrization node

For each effect, describe what you see and hypothesize a reason why. Add your answers to this notebook.

                                                                                                                                              

In [ ]:
## Experiments

There are a number of critical parameters here in this notebook. Explore what happens when you make the following changes and train the model (you probably won't need all 20 epochs to see a result)

1. Set $\beta = 0$ and then $\beta = 1$
                                                                                                                                               2.
